# Hub effect exploration

For a random sample of clue-vocabulary words, compute each clue's mean
cosine similarity to the full board vocabulary (`sims.board_words`), per
embedding space, and plot the distribution.

Context: `docs/versions/v1.md` found that a clue's *typical* similarity to
the board vocabulary correlates strongly with word frequency (r=0.84 for
GloVe) -- some words sit at high similarity to almost everything
regardless of real semantic relatedness (the "hubness" effect in cosine-
similarity spaces). This notebook just looks at the raw distribution per
space, as a first look before checking that correlation directly.

Requires `cache/similarity_tensor.npy` to already be built
(`scripts/build_similarity_tensor.py`).

In [ ]:
import sys
from pathlib import Path

# Notebook lives in scratch/, one level below the project root.
PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "codenames").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from codenames.similarity import SimilarityTensor

In [ ]:
N_SAMPLES = 2000
SEED = 0

In [ ]:
sims = SimilarityTensor.load()
print(f"{len(sims.clue_words)} clues, {len(sims.board_words)} board words, spaces: {sims.spaces}")

In [ ]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(sims.clue_words), size=N_SAMPLES, replace=False)
sample_clues = [sims.clue_words[i] for i in sample_idx]

In [ ]:
# tensor[sample_idx, :, space_idx] -> (N_SAMPLES, n_board_words); mean
# across board words gives one "typical similarity" value per sampled
# clue, per space.
mean_sim_by_space = {}
for space_idx, space in enumerate(sims.spaces):
    values = np.asarray(sims.tensor[sample_idx, :, space_idx], dtype=np.float32)
    with np.errstate(invalid="ignore"):
        mean_sim_by_space[space] = np.nanmean(values, axis=1)

In [ ]:
fig, axes = plt.subplots(1, len(sims.spaces), figsize=(5 * len(sims.spaces), 4), sharey=True)
for ax, space in zip(axes, sims.spaces):
    ax.hist(mean_sim_by_space[space], bins=50)
    ax.set_title(space)
    ax.set_xlabel("mean cosine similarity to board vocabulary")
axes[0].set_ylabel("count")
fig.suptitle(f"Per-clue mean similarity to board vocabulary ({N_SAMPLES} sampled clues)")
fig.tight_layout()
plt.show()

## Per-word similarity distributions

The plots above collapse each clue down to one summary number (its mean
similarity across the board vocabulary), which can't distinguish "high
similarity to a few genuinely related words" from "uniformly medium-high
similarity to almost everything" (the hub signature). This section looks
at a handful of individual sample words directly -- for each one, the full
distribution of its similarity to every board-vocabulary word -- to see
whether that per-word spread actually varies much word to word, or space
to space.

In [ ]:
N_WORDS = 6
word_idx = rng.choice(len(sims.clue_words), size=N_WORDS, replace=False)
sample_words = [sims.clue_words[i] for i in word_idx]
sample_words

In [ ]:
fig, axes = plt.subplots(N_WORDS, len(sims.spaces), figsize=(5 * len(sims.spaces), 3 * N_WORDS), sharex=True)
for row, (word, ci) in enumerate(zip(sample_words, word_idx)):
    for col, space in enumerate(sims.spaces):
        ax = axes[row, col]
        values = np.asarray(sims.tensor[ci, :, col], dtype=np.float32)
        values = values[~np.isnan(values)]
        ax.hist(values, bins=50)
        if row == 0:
            ax.set_title(space)
        if col == 0:
            ax.set_ylabel(word)
fig.suptitle("Per-word similarity to every board-vocabulary word")
fig.tight_layout()
plt.show()